# G1, Replay-IS + DAgger: kiểm tra bản tối ưu kỹ thuật (Kaggle, GPU)

Bản code này **không đổi thuật toán** (cùng số env, cùng dữ liệu cũ, cùng lịch gắn nhãn), chỉ chạy
nhanh hơn ở ba chỗ:

1. Dữ liệu cũ (4 đoạn) được đưa qua actor và critic **một lần** thay vì từng đoạn một.
2. Khi MPC thử các chuỗi hành động, môi trường lập kế hoạch **không tính quan sát** (chỉ cần reward).
3. Mỗi vòng ghi thời gian từng giai đoạn: `time/mpc`, `time/replay`, `time/sgd`, `time/rest`.

Kiểm tra trên máy: kết quả của (1) và reward của (2) trùng với bản cũ; mọi test đều qua.

Notebook train **Replay-IS+DAgger** (cấu hình gốc) với 2 seed, mỗi seed trên một GPU, rồi so với lần
chạy trước trên Kaggle (cùng cấu hình, bản code cũ):

| | Tổng thời gian | s/vòng | Phút tới "bám tốt" | Vận tốc ở 1,5 m/s | Sai số |
|---|---|---|---|---|---|
| Bản cũ (lần chạy trước) | 122 phút | 3,68 | 75 | 1,49 | 0,039 |
| PPO (lần chạy trước) | 92 phút | 2,77 | 78 | 1,47 | 0,048 |

Bật `TRAIN["PPO"]` nếu muốn đo lại PPO trong cùng phiên (thêm khoảng 1,5 giờ).

## Cài đặt Kaggle trước khi chạy (panel bên phải → *Settings*)

1. **Accelerator: GPU T4 x2.** Không chọn P100: bản PyTorch của dự án (CUDA 12.8) không hỗ trợ P100.
2. **Internet: On** (cần xác minh số điện thoại cho tài khoản Kaggle) để tải code và thư viện.

## Cách chạy

**Save Version → Save & Run All (Commit)**. Mặc định 2 lần train song song ≈ 2–2,5 giờ. Kết quả ở tab
**Output**: `mpc_g1_opt_results.zip`.

## 0. Cấu hình

In [ ]:
import os
import time

NOTEBOOK_START = time.time()

REPO_URL = "https://github.com/TamasTran/mjlab_MPOPI.git"
BRANCH = "mpc-stage1"
TASK = "Mjlab-Velocity-Flat-Unitree-G1-2k"
ITERATIONS = 2000
SEEDS = [1, 2]

# Cấu hình cần train (True/False).
TRAIN = {
  "PPO": False,
  "Replay-IS+DAgger": True,
}
MIN_ACTION_STD = 0.05  # Mức sàn std của policy, dùng chung cho mọi phương pháp.

# MPC (thầy): env phụ × bước mỗi lần thu, thu mỗi MPC_EVERY vòng trong MPC_ITERATIONS vòng đầu.
MPC_ENVS, MPC_STEPS, MPC_EVERY, MPC_ITERATIONS = 64, 24, 5, 110
PLANNER_K, PLANNER_L, PLANNER_H = 16, 2, 16  # MPOPI: K mẫu × L vòng, tầm nhìn H bước.
# Đo thầy có hơn trò không (Δ): cứ mỗi TEACHER_GAP_EVERY bước gắn nhãn, so return của kế
# hoạch MPC với return của policy từ cùng trạng thái. None để tắt (thêm ~12% chi phí MPC).
TEACHER_GAP_EVERY = None

# Canh giờ (Kaggle dừng phiên GPU sau khoảng 12 giờ).
TIME_BUDGET_HOURS = 11.5  # Tính từ lúc chạy ô này.
EVAL_RESERVE_MIN = 60  # Phút chừa lại cho đánh giá, video và đóng gói.
SMOKE_TEST = True  # Chạy thử 3 vòng mỗi phương pháp và ước tính thời gian.
EVAL_ENVS = 16
RECORD_VIDEO = True

REPO_DIR = "/tmp/mjlab_MPOPI"  # Không để trong /kaggle/working (thư mục output).
OUT_ROOT = "/kaggle/working/mpc_g1_opt"
LOG_ROOT = f"{OUT_ROOT}/train_logs"
CONSOLE_DIR = f"{OUT_ROOT}/train_console"
EVAL_DIR = f"{OUT_ROOT}/eval"
for d in (CONSOLE_DIR, EVAL_DIR):
  os.makedirs(d, exist_ok=True)
DEADLINE = NOTEBOOK_START + TIME_BUDGET_HOURS * 3600
TRAIN_DEADLINE = DEADLINE - EVAL_RESERVE_MIN * 60
N_SEGMENTS = -(-MPC_ITERATIONS // MPC_EVERY)


def remaining_min(deadline=DEADLINE):
  return (deadline - time.time()) / 60


print(f"Thời gian cho train: {remaining_min(TRAIN_DEADLINE):.0f} phút; tổng: {remaining_min():.0f} phút")

In [ ]:
#@title Hàm tiện ích
import subprocess


def run(cmd, cwd=None, quiet_tables=False, check=True):
  """Run a shell command and stream its output."""
  print(f"$ {cmd}", flush=True)
  proc = subprocess.Popen(
    cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env={**os.environ},
  )
  for line in proc.stdout:
    if quiet_tables and ("[INFO]" in line or line.startswith(("|", "+"))):
      continue
    print(line, end="", flush=True)
  if proc.wait() != 0 and check:
    raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
  return proc.returncode

## 1. Kiểm tra GPU

In [ ]:
gpu_list = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu_list)
GPUS = [line.split(":")[0].split()[-1] for line in gpu_list.splitlines() if line.startswith("GPU")]
if not GPUS:
  raise RuntimeError("Không thấy GPU. Settings → Accelerator → GPU T4 x2.")
if "P100" in gpu_list:
  raise RuntimeError("P100 không được hỗ trợ. Settings → Accelerator → GPU T4 x2.")
print(f"Dùng {len(GPUS)} GPU: {GPUS}")

## 2. Chép kết quả phiên trước (nếu có) và lấy code

In [ ]:
import glob
import shutil

previous = sorted(glob.glob("/kaggle/input/**/mpc_g1_opt", recursive=True))
for prev in previous:
  print("Chép kết quả phiên trước từ", prev)
  shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
done_before = sorted(os.path.basename(p)[:-5] for p in glob.glob(f"{CONSOLE_DIR}/*.done"))
print("Các lần train đã xong:", done_before or "chưa có")

if not REPO_DIR.startswith("/tmp/"):
  raise RuntimeError("REPO_DIR phải nằm trong /tmp (ô này xóa rồi clone lại thư mục đó).")
if os.path.exists(REPO_DIR):
  shutil.rmtree(REPO_DIR)
run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
run("git log --oneline -1", cwd=REPO_DIR)
init_src = open(f"{REPO_DIR}/src/mjlab/tasks/velocity/config/g1/__init__.py").read()
config_src = open(f"{REPO_DIR}/src/mjlab/rl/mpopi/config.py").read()
if TASK not in init_src or "teacher_gap_every" not in config_src:
  raise RuntimeError("Code trên GitHub đã cũ: hãy push nhánh mpc-stage1 bản mới nhất.")
print("OK")

## 3. Cài đặt môi trường

Lần đầu mất khoảng 5–10 phút.

In [ ]:
run("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:{os.environ['PATH']}"
run("uv sync --extra cu128", cwd=REPO_DIR)
run(
  "uv run --extra cu128 python -c \"import torch; print('torch', torch.__version__, "
  "'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))\"",
  cwd=REPO_DIR,
)
UV = "uv run --extra cu128"

## 4. Lệnh train của từng phương pháp

In [ ]:
A = "--agent.algorithm.mpopi"
COMMON = f"{A}.min-action-std {MIN_ACTION_STD}"
MPC_TEACHER = (
  f"{A}.mode mpc_ppo {A}.mpc.num-envs {MPC_ENVS} {A}.mpc.num-steps {MPC_STEPS} "
  f"{A}.mpc.collect-every {MPC_EVERY} {A}.mpc.collect-iterations {MPC_ITERATIONS} "
  f"{A}.mpc.execution-std 0 {A}.mpc.planner.num-samples {PLANNER_K} "
  f"{A}.mpc.planner.iterations {PLANNER_L} {A}.mpc.planner.horizon {PLANNER_H} "
  f"{A}.mpc.planner.noise-std 0.2 {A}.mpc.planner.num-knots 4"
)
DAGGER = (
  f"{MPC_TEACHER} {A}.mpc.use-in-ppo False {A}.mpc.driver policy "
  f"{A}.mpc.bc-coef 1.0 {A}.mpc.bc-iterations {MPC_ITERATIONS + 40} "
  f"{A}.mpc.max-age 50 {A}.mpc.buffer-segments 10 {A}.mpc.replay-own-rollouts True"
  + (f" {A}.mpc.teacher-gap-every {TEACHER_GAP_EVERY}" if TEACHER_GAP_EVERY else "")
)
# Mỗi cấu hình: (số env chính, tùy chọn). Đổi số env phụ MPC bằng cách ghi đè num-envs.
ALL_METHODS = {
  "PPO": (4096, f"{A}.mode ppo"),
  "Replay-IS+DAgger": (4096, DAGGER),
}
METHODS = {m: a for m, a in ALL_METHODS.items() if TRAIN[m]}
NUM_ENVS_OF = {m: METHODS[m][0] for m in METHODS}


def train_cmd(method, seed, iterations=ITERATIONS, log_root=LOG_ROOT):
  num_envs, args = METHODS[method]
  return (
    f"{UV} train {TASK} --agent.logger tensorboard --env.scene.num-envs {num_envs} "
    f"--agent.max-iterations {iterations} --agent.seed {seed} "
    f"--agent.run-name {method}_s{seed} --log-root {log_root} {COMMON} {args}"
  )


for m in METHODS:
  print(f"# {m}\n{train_cmd(m, SEEDS[0])}\n")

## 5. Chạy thử và ước tính thời gian

Mỗi cấu hình chạy 3 vòng với số env của nó (thu MPC ở cả 3 vòng) để kiểm tra lỗi và đo thời gian
một vòng PPO và một lần thu MPC. Từ đó ước tính thời gian của cả lần train.

In [ ]:
import re

ANSI = re.compile(r"\x1b\[[0-9;]*m")
ESTIMATE_S = {}  # Ước tính thời gian (giây) một lần train đầy đủ của mỗi phương pháp.
if SMOKE_TEST:
  for m in METHODS:
    log = f"{OUT_ROOT}/smoke_{m}.log"
    cmd = train_cmd(m, 0, iterations=3, log_root="/tmp/smoke_logs")
    cmd = cmd.replace(f"collect-every {MPC_EVERY} ", "collect-every 1 ")
    t0 = time.time()
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": GPUS[0]}
    rc = subprocess.run(f"{cmd} > '{log}' 2>&1", shell=True, cwd=REPO_DIR, env=env).returncode
    text = ANSI.sub("", open(log, errors="ignore").read())
    if rc != 0:
      print(text[-3000:])
      raise RuntimeError(f"Chạy thử {m} bị lỗi")
    it_times = [float(x) for x in re.findall(r"Iteration time:\s*([\d.]+)s", text)]
    mpc_s = [float(x) for x in re.findall(r"mpc/collect_seconds loss:\s*([\d.]+)", text)]
    per_it = min(it_times) if it_times else 0.0
    if mpc_s and len(mpc_s) == len(it_times):
      per_it = max(0.0, per_it - min(mpc_s))  # Mỗi vòng chạy thử đều có thu MPC.
    est = 1.1 * per_it * ITERATIONS + (N_SEGMENTS * max(mpc_s) if mpc_s else 0.0)
    ESTIMATE_S[m] = est
    info = f"một vòng ~{per_it:.1f} s" + (f", một lần thu MPC ~{max(mpc_s):.0f} s" if mpc_s else "")
    print(f"OK {m}: chạy thử {time.time() - t0:.0f} s ({info}) → ước tính cả lần train "
          f"{est / 3600:.1f} giờ", flush=True)
  jobs_h = sum(ESTIMATE_S[m] for m in METHODS) * len(SEEDS) / 3600
  print(f"\nTổng ≈ {jobs_h:.1f} giờ GPU; với {len(GPUS)} GPU song song ≈ {jobs_h / len(GPUS):.1f} giờ. "
        f"Thời gian còn cho train: {remaining_min(TRAIN_DEADLINE) / 60:.1f} giờ.")

## 6. Train

Mỗi GPU lấy lần lượt một lần train từ hàng đợi. Lần train nào ước tính không kịp xong trước giờ
chót thì bị bỏ qua (chạy ở phiên sau); lần đang chạy mà tới giờ chót thì bị dừng. Tiến độ được in
mỗi 10 phút.

In [ ]:
import queue
import threading

jobs = queue.Queue()
for s in SEEDS:
  for m in METHODS:
    if not os.path.exists(f"{CONSOLE_DIR}/{m}_s{s}.done"):
      jobs.put((m, s))
STATUS = {}


def lane(gpu):
  while True:
    try:
      m, s = jobs.get_nowait()
    except queue.Empty:
      return
    name = f"{m}_s{s}"
    est = ESTIMATE_S.get(m)
    if est is not None and time.time() + est > TRAIN_DEADLINE:
      STATUS[name] = f"bỏ qua (cần ~{est / 3600:.1f} giờ, không đủ thời gian)"
      continue
    STATUS[name] = f"đang chạy trên GPU {gpu}"
    log = f"{CONSOLE_DIR}/{name}.log"
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu, "PYTHONUNBUFFERED": "1"}
    t0 = time.time()
    with open(log, "w") as f:
      p = subprocess.Popen(train_cmd(m, s), shell=True, cwd=REPO_DIR, env=env,
                           stdout=f, stderr=subprocess.STDOUT, start_new_session=True)
      while p.poll() is None:
        if time.time() > TRAIN_DEADLINE:
          os.killpg(p.pid, 15)
          p.wait()
          break
        time.sleep(20)
    if p.returncode == 0:
      open(f"{CONSOLE_DIR}/{name}.done", "w").write(f"{time.time() - t0:.0f}\n")
      STATUS[name] = f"XONG sau {(time.time() - t0) / 3600:.2f} giờ"
    elif time.time() > TRAIN_DEADLINE:
      STATUS[name] = "dừng vì hết giờ (train lại ở phiên sau)"
    else:
      STATUS[name] = f"LỖI (mã {p.returncode}), xem {log}"


def last_progress(name):
  path = f"{CONSOLE_DIR}/{name}.log"
  if not os.path.exists(path):
    return ""
  text = ANSI.sub("", open(path, errors="ignore").read())
  its = re.findall(r"Learning iteration\s+(\d+)/", text)
  eta = re.findall(r"ETA:\s*([\d:]+)", text)
  return f"vòng {its[-1]}/{ITERATIONS}, còn {eta[-1] if eta else '?'}" if its else "đang khởi tạo"


threads = [threading.Thread(target=lane, args=(g,), daemon=True) for g in GPUS]
for t in threads:
  t.start()
last_print = 0.0
while any(t.is_alive() for t in threads):
  time.sleep(20)
  if time.time() - last_print > 600:
    last_print = time.time()
    print(time.strftime("%H:%M"), f"| còn {remaining_min(TRAIN_DEADLINE):.0f} phút cho train", flush=True)
    for name, st in sorted(STATUS.items()):
      extra = f" ({last_progress(name)})" if st.startswith("đang chạy") else ""
      print(f"  {name:>20}: {st}{extra}", flush=True)
print("\nKết thúc train:")
for name, st in sorted(STATUS.items()):
  print(f"  {name:>20}: {st}")

## 7. Đường học

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

COLORS = {"PPO": "#555555", "Replay-IS+DAgger": "#9467bd", "Replay-IS+DAgger-A": "#1f77b4",
          "Replay-IS+DAgger-B": "#ff7f0e"}
TAGS = {
  "Train/mean_reward": "Mean reward (lúc train)",
  "Episode_Reward/track_linear_velocity": "Bám vận tốc thẳng (tối đa 2,0)",
  "Train/mean_episode_length": "Độ dài episode (bước)",
}


def done(m, s):
  return os.path.exists(f"{CONSOLE_DIR}/{m}_s{s}.done")


def run_dir(m, s):
  found = glob.glob(f"{LOG_ROOT}/**/*_{m}_s{s}", recursive=True)
  return max(found, key=os.path.getmtime) if found else None


def load_scalars(path):
  ea = EventAccumulator(path, size_guidance={"scalars": 0})
  ea.Reload()
  have = ea.Tags()["scalars"]
  return {t: {e.step: e.value for e in ea.Scalars(t)} if t in have else {} for t in TAGS}


fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for m in METHODS:
  curves = {t: [] for t in TAGS}
  for s in SEEDS:
    d = run_dir(m, s)
    if d and done(m, s):
      sc = load_scalars(d)
      for t in TAGS:
        if sc[t]:
          curves[t].append(sc[t])
  for ax, t in zip(axes, TAGS):
    cs = curves[t]
    for c in cs:
      ks = sorted(c)
      ax.plot(ks, [c[k] for k in ks], color=COLORS[m], alpha=0.25, lw=1)
    if cs:
      common = sorted(set.intersection(*(set(c) for c in cs)))
      ax.plot(common, [np.mean([c[k] for c in cs]) for k in common], color=COLORS[m], lw=2,
              label=f"{m} ({len(cs)} seed)")
for ax, t in zip(axes, TAGS):
  ax.axvline(500, color="k", ls="--", lw=0.8)
  ax.set_xlabel("Vòng lặp")
  ax.set_ylabel(TAGS[t])
  ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/train_curves.png", dpi=150)
plt.show()

## 8. Đo vận tốc thực tế ở 0,5 / 1,0 / 1,5 m/s

Checkpoint cuối của mỗi lần train đã xong; policy tất định điều khiển 16 robot đi thẳng 5 giây ở
từng lệnh, bỏ 1 giây đầu. **Tiêu chí "bám 1,5 m/s":** vận tốc trung bình ≥ 1,2 m/s ở lệnh 1,5 m/s
và ≤ 0,25 lần ngã mỗi robot.

In [ ]:
def last_checkpoint(d):
  ckpts = glob.glob(f"{d}/model_*.pt") if d else []
  return max(ckpts, key=lambda p: int(re.search(r"model_(\d+)", p).group(1))) if ckpts else None


first = not glob.glob(f"{EVAL_DIR}/*.json")
for s in SEEDS:
  for m in METHODS:
    out = f"{EVAL_DIR}/{m}_s{s}.json"
    ckpt = last_checkpoint(run_dir(m, s))
    if os.path.exists(out) or not done(m, s) or ckpt is None:
      continue
    if remaining_min() < 10:
      print("Gần hết giờ, dừng đánh giá.")
      break
    controllers = "zero policy" if first else "policy"
    first = False
    print(f"===== {m}_s{s}: {os.path.basename(ckpt)}", flush=True)
    run(
      f"CUDA_VISIBLE_DEVICES={GPUS[0]} {UV} python scripts/mpc/eval_mpc_velocity.py --task {TASK} "
      f"--device cuda:0 --num-envs {EVAL_ENVS} --steps 250 --speeds 0.5 1.0 1.5 "
      f"--controllers {controllers} --checkpoint {ckpt} --out {out}",
      cwd=REPO_DIR, quiet_tables=True, check=False,
    )

In [ ]:
import json

rows = []
for path in sorted(glob.glob(f"{EVAL_DIR}/*.json")):
  name = os.path.basename(path)[:-5]
  method = name.rsplit("_s", 1)[0]
  for r in json.load(open(path))["results"]:
    rows.append({"method": method if r["controller"] == "policy" else "đứng yên", "run": name, **r})

lines = [f"{'phương pháp':>18} {'lệnh':>5} {'vận tốc':>14} {'|sai số|':>9} {'ngã/robot':>10} {'seed':>5}"]
summary = {}
for method in ["đứng yên", *METHODS]:
  for cmd in (0.5, 1.0, 1.5):
    rs = [r for r in rows if r["method"] == method and r["command"] == cmd]
    if not rs:
      continue
    v = np.array([r["speed"] for r in rs])
    spread = f" ± {v.std():.2f}" if len(v) > 1 else ""
    lines.append(f"{method:>18} {cmd:>5.1f} {v.mean():>8.2f}{spread:<6} "
                 f"{np.mean([r['abs_error'] for r in rs]):>9.2f} "
                 f"{np.mean([r['falls_per_env'] for r in rs]):>10.2f} {len(rs):>5}")
    if cmd == 1.5 and method != "đứng yên":
      summary[method] = [(r["run"], r["speed"], r["falls_per_env"]) for r in rs]
lines.append("\nTiêu chí bám 1,5 m/s (≥ 1,2 m/s và ≤ 0,25 lần ngã/robot):")
for method, runs in summary.items():
  ok = [run_name for run_name, v, f in runs if v >= 1.2 and f <= 0.25]
  lines.append(f"  {method}: {len(ok)}/{len(runs)} lần train ĐẠT")
report = "\n".join(lines)
print(report)
open(f"{OUT_ROOT}/ket_qua.txt", "w").write(report + "\n")

## 9. Bảng tổng hợp: thời gian và tốc độ học

Trung bình theo seed. Mốc tính trên trung bình trượt 10 vòng: *đứng vững* (episode ≥ 900 bước),
*bám vận tốc* / *bám tốt* (reward bám vận tốc thẳng ≥ 1,0 / ≥ 1,5 trên tối đa 2). Thời gian tới mốc
cộng dồn thời gian thu dữ liệu và cập nhật từ vòng 0. Bước env = vòng × số env × 24.

In [ ]:
def first_hit(values, th, k=10):
  v = np.asarray(values, dtype=float)
  if len(v) < k:
    return None
  sm = np.convolve(v, np.ones(k) / k, mode="valid")
  idx = np.where(sm >= th)[0]
  return int(idx[0]) + k - 1 if len(idx) else None


MILES = [("Train/mean_episode_length", 900, "đứng vững"),
         ("Episode_Reward/track_linear_velocity", 1.0, "bám vận tốc"),
         ("Episode_Reward/track_linear_velocity", 1.5, "bám tốt")]
table = {}
for m in METHODS:
  for s in SEEDS:
    d = run_dir(m, s)
    if not (d and done(m, s)):
      continue
    ea = EventAccumulator(d, size_guidance={"scalars": 0})
    ea.Reload()
    get = lambda t: np.array([e.value for e in ea.Scalars(t)])
    per_it = get("Perf/collection_time") + get("Perf/learning_time")
    cum = np.cumsum(per_it)
    hits = [first_hit(get(t), th) for t, th, _ in MILES]
    ev = [r for r in rows if r["run"] == f"{m}_s{s}" and r["command"] == 1.5 and r["method"] != "đứng yên"]
    table.setdefault(m, []).append({
      "total_min": float(open(f"{CONSOLE_DIR}/{m}_s{s}.done").read()) / 60,
      "s_per_it": per_it.mean(),
      "its": hits,
      "mins": [cum[h] / 60 if h is not None else np.nan for h in hits],
      "steps_m": [h * NUM_ENVS_OF[m] * 24 / 1e6 if h is not None else np.nan for h in hits],
      "speed": ev[0]["speed"] if ev else np.nan,
      "err": ev[0]["abs_error"] if ev else np.nan,
    })

fmt = lambda xs: " / ".join("--" if (x is None or np.isnan(x)) else f"{x:.0f}" for x in xs)
lines = [f"{'cấu hình':>20} {'seed':>4} {'tổng (phút)':>11} {'s/vòng':>7} "
         f"{'vòng tới 3 mốc':>18} {'phút tới 3 mốc':>16} {'vận tốc 1,5':>11} {'|e|':>6}"]
for m, rs in table.items():
  mean = lambda key: np.nanmean([r[key] for r in rs], axis=0)
  its = [np.nan if any(r["its"][i] is None for r in rs) else np.mean([r["its"][i] for r in rs])
         for i in range(3)]
  lines.append(f"{m:>20} {len(rs):>4} {mean('total_min'):>11.0f} {mean('s_per_it'):>7.2f} "
               f"{fmt(its):>18} {fmt(mean('mins')):>16} {mean('speed'):>11.2f} {mean('err'):>6.3f}")
summary_text = "\n".join(lines)
print(summary_text)
open(f"{OUT_ROOT}/bang_thoi_gian.txt", "w").write(summary_text + "\n")

## 9b. Thời gian từng giai đoạn của mỗi vòng

`thu` = mô phỏng env chính (như PPO); trong phần cập nhật: `mpc` = gắn nhãn MPC (chỉ ở các vòng gắn
nhãn), `replay` = chuẩn bị dữ liệu cũ và nhãn MPC, `sgd` = các bước gradient, `rest` = chuẩn hóa
quan sát và lưu đoạn mới. Đơn vị: giây/vòng (trung vị) và tổng số phút cả lần train.

In [ ]:
PHASES = ["mpc", "replay", "sgd", "rest"]
lines = [f"{'lần train':>24} {'thu':>12} " + " ".join(f"{p:>14}" for p in PHASES)]
for m in METHODS:
  for s in SEEDS:
    d = run_dir(m, s)
    if not (d and done(m, s)):
      continue
    ea = EventAccumulator(d, size_guidance={"scalars": 0})
    ea.Reload()
    tags = ea.Tags()["scalars"]
    get = lambda t: np.array([e.value for e in ea.Scalars(t)]) if t in tags else None
    cols = [get("Perf/collection_time")] + [get(f"Loss/time/{p}") for p in PHASES]
    cell = lambda v: "--" if v is None else f"{np.median(v):5.2f} s {v.sum() / 60:4.0f}'"
    lines.append(f"{m + '_s' + str(s):>24} {cell(cols[0]):>12} " + " ".join(f"{cell(v):>14}" for v in cols[1:]))
phase_text = "\n".join(lines)
print(phase_text)
open(f"{OUT_ROOT}/thoi_gian_giai_doan.txt", "w").write(phase_text + "\n")

## 10. Video ở lệnh 1,5 m/s (seed đầu tiên)

Nếu máy không quay được video (thiếu EGL), ô này chỉ báo lỗi, không ảnh hưởng kết quả.

In [ ]:
from IPython.display import HTML, Video, display

if RECORD_VIDEO:
  for m in METHODS:
    s = SEEDS[0]
    vdir = f"{EVAL_DIR}/video/{m}_s{s}"
    ckpt = last_checkpoint(run_dir(m, s))
    if not done(m, s) or ckpt is None or glob.glob(f"{vdir}/*.mp4") or remaining_min() < 5:
      continue
    run(
      f"CUDA_VISIBLE_DEVICES={GPUS[0]} {UV} python scripts/mpc/eval_mpc_velocity.py --task {TASK} "
      f"--device cuda:0 --num-envs 1 --steps 250 --speeds 1.5 --controllers policy "
      f"--checkpoint {ckpt} --video-dir {vdir}",
      cwd=REPO_DIR, quiet_tables=True, check=False,
    )
for path in sorted(glob.glob(f"{EVAL_DIR}/video/*/*.mp4")):
  display(HTML(f"<b>{os.path.relpath(path, EVAL_DIR)}</b>"))
  display(Video(path, embed=True, width=480))

## 11. Dọn và đóng gói

Chỉ giữ checkpoint cuối của mỗi lần train (để phiên sau đánh giá lại được) và nén kết quả thành
`/kaggle/working/mpc_g1_opt_results.zip` (không kèm checkpoint). Tải về ở tab **Output**.

In [ ]:
import zipfile

for d in glob.glob(f"{LOG_ROOT}/**/", recursive=True):
  ckpts = glob.glob(f"{d}/model_*.pt")
  keep = last_checkpoint(d) if ckpts else None
  for c in ckpts:
    if c != keep:
      os.remove(c)

src = "/kaggle/working/mpc_g1_opt_results.zip"
with zipfile.ZipFile(src, "w", zipfile.ZIP_DEFLATED) as zf:
  for root, _, files in os.walk(OUT_ROOT):
    for fname in files:
      if not fname.endswith((".pt", ".onnx")):
        path = os.path.join(root, fname)
        zf.write(path, os.path.relpath(path, OUT_ROOT))
print(f"Xong. Kích thước zip: {os.path.getsize(src) / 1e6:.1f} MB")
print(f"Tổng thời gian: {(time.time() - NOTEBOOK_START) / 3600:.2f} giờ")